---슈도 코드  
크로마 DB 접속  
크로마 DB에 테이블 생성 
트레이닝 데이터 중 질문만 백터화하고 나머지는 메타 데이터로 입력(시스템 프롬프트는 제외)  
오픈 ai LLM 연결  
벡터 유사도 검색 테스트  
시간 남으면 평가 데이터로 질문 정확도 테스트(평가 데이터 중 골라서 질문을 넣기or트레이닝 데이터랑 유사한 질문을 조원들끼리 나눠서 생성하고 알맞은 답변을 골라놓기)  


In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한국어 임베딩 모델을 준비합니다.
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb

# 지난 단원에서 배운 한국어 임베딩 모델 — 문장 한 개를 768차원 벡터로 바꿉니다.
# (처음 부를 때 모델을 내려받느라 조금 걸릴 수 있어요.)
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 — 벡터 차원:', emb_model.get_embedding_dimension())

In [ ]:
df = pd.read_csv('../data/training/df.csv')
print(df)

In [ ]:
df = df.drop(labels='qa.instruction',axis=1)

In [ ]:
print(df.head())

In [ ]:
print(df.shape)
print(df['meta.disease'].value_counts())

In [ ]:
df.rename(columns = {'Unnamed: 0' : 'ids'}, inplace = True)

In [ ]:
df.head()

In [ ]:
# 1) 인덱싱 타임 — 모든 여행지 소개를 임베딩(문서 개수 × 768차원)
doc_texts = df['qa.input'].tolist()
doc_emb = emb_model.encode(doc_texts, normalize_embeddings=True)

print("문서 임베딩 행렬:", doc_emb.shape)

# 1.0 에 아주 가까우면 길이가 1로 잘 맞춰진 것이다
print("첫 문서 벡터의 길이:", round(float(np.linalg.norm(doc_emb[0])), 4))

In [ ]:
df['ids'] = df.ids.astype(str)

In [ ]:
df.info()

In [ ]:
print(len(df))
print(len(doc_emb))

In [ ]:
# 1) 클라이언트 — 메모리에 살아서 커널을 끄면 사라진다
client = chromadb.PersistentClient(path='./chroma_db')

# 2) 컬렉션 — get_or_create 라 여러 번 실행해도 안전하다
pet_care_col = client.get_or_create_collection(
    'pet_care',
    metadata={'hnsw:space': 'cosine'},   # 거리 기준
)

# 3) 적재 — 네 인자는 같은 위치끼리 한 문서를 이룬다
pet_care_col.add(
    ids=df['ids'].tolist(),
    embeddings=doc_emb,                        # 우리 벡터를 직접 넣는다
    documents=df['qa.input'].tolist(),
    metadatas=[
        {'meta.lifeCycle': n, 'meta.department': r, 'meta.disease': t, 'qa.output': f}
        for n, r, t, f in zip(
            df['meta.lifeCycle'], df['meta.department'], df['meta.disease'], df['qa.output']
        )
    ],
)

print("컬렉션에 저장된 문서 수:", pet_care_col.count())

# 4) get 은 검색이 아니라 id 로 직접 꺼내는 것
peek = pet_care_col.get(ids=['0'])

print('원문:', peek['documents'][0])
print('메타데이터:', peek['metadatas'][0])

In [ ]:
import chromadb

# 1) ChromaDB 클라이언트
client = chromadb.PersistentClient(path='./chroma_db')

# 2) 컬렉션
pet_care_col = client.get_or_create_collection(
    'pet_care',
    metadata={'hnsw:space': 'cosine'}
)

# 3) 데이터 나눠서 적재
BATCH_SIZE = 5000
total = len(df)

for start in range(0, total, BATCH_SIZE):

    end = min(start + BATCH_SIZE, total)

    pet_care_col.add(
        ids=df['ids'].iloc[start:end].astype(str).tolist(),

        embeddings=doc_emb[start:end],

        documents=df['qa.input'].iloc[start:end].tolist(),

        metadatas=[
            {
                'meta.lifeCycle': n,
                'meta.department': r,
                'meta.disease': t,
                'qa.output': f
            }
            for n, r, t, f in zip(
                df['meta.lifeCycle'].iloc[start:end],
                df['meta.department'].iloc[start:end],
                df['meta.disease'].iloc[start:end],
                df['qa.output'].iloc[start:end]
            )
        ],
    )

    print(f"{end}/{total}개 저장 완료")


# 4) 저장 결과 확인
print("컬렉션에 저장된 문서 수:", pet_care_col.count())


# 5) ID로 데이터 확인
peek = pet_care_col.get(ids=['0'])

print('원문:', peek['documents'][0])
print('메타데이터:', peek['metadatas'][0])